In [2]:
import numpy as np
import pandas as pd
from rdkit import Chem, DataStructs
from rdkit.Chem import Descriptors, rdFingerprintGenerator

gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=1024)
mol = Chem.MolFromSmiles("CN(C)C(=N)NC(N)=N")  # metformin
fp = gen.GetFingerprintAsNumPy(mol)  # 1024 bits; 17 are 1
print(Descriptors.MolWt(mol), Descriptors.MolLogP(mol))  # 129.17, -1.03

aspirin = Chem.MolFromSmiles("CC(=O)Oc1ccccc1C(=O)O")
print(
    DataStructs.TanimotoSimilarity(
        gen.GetFingerprint(mol),
        gen.GetFingerprint(aspirin),
    )
)  # 0.05


def trial_fingerprint(smiles_field):
    """Combine the fingerprints of all matched drugs in one trial."""
    if pd.isna(smiles_field):
        return np.zeros(1024)  # no matched drug
    fps = [
        gen.GetFingerprintAsNumPy(m)
        for s in smiles_field.split(";")
        if (m := Chem.MolFromSmiles(s)) is not None
    ]
    return np.max(fps, axis=0) if fps else np.zeros(1024)


train = pd.read_csv("Part 1/part1_train.csv")
X_fp = np.vstack(train["drug_smiles"].apply(trial_fingerprint))  # (1171, 1024)

129.167 -1.034159999999999
0.05128205128205128
